# V8.0: Two-Stage Training (BraTS2021 Pretrain + TextBraTS Fine-tune)

**Stage 1:** BraTS2021 (1251 cases, no text) — learn visual features
**Stage 2:** BraTS2020 (295 cases + TextBraTS text) — add text guidance

In [ ]:
# ===== Setup =====
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi 2>/dev/null || echo 'No GPU'

!pip install -q --cache-dir=/content/drive/MyDrive/pip_cache \
    mamba-ssm causal-conv1d einops \
    transformers nibabel pyyaml tqdm scipy

import os, subprocess, zipfile, time, shutil, glob
REPO_DIR = '/content/TextMamba3D'
DRIVE_BASE = '/content/drive/MyDrive/TextMamba3D'
DRIVE_CKPT = os.path.join(DRIVE_BASE, 'checkpoints')
os.makedirs(DRIVE_CKPT, exist_ok=True)

git_dir = os.path.join(REPO_DIR, '.git')
if os.path.isdir(REPO_DIR) and not os.path.isdir(git_dir):
    shutil.rmtree(REPO_DIR)
if os.path.isdir(git_dir):
    os.chdir(REPO_DIR)
    subprocess.run(['git', 'pull'], check=True)
else:
    for attempt in range(1, 4):
        ret = subprocess.run(
            ['git', 'clone', '--depth', '1',
             'https://github.com/PlutoLei/TextMamba3D.git', REPO_DIR],
            capture_output=True, text=True)
        if ret.returncode == 0:
            break
        print(f'Clone attempt {attempt} failed')
        if os.path.isdir(REPO_DIR):
            shutil.rmtree(REPO_DIR)
        time.sleep(5 * attempt)
    else:
        raise RuntimeError('Clone failed')
    os.chdir(REPO_DIR)

# Unzip BraTS2020 + TextBraTS data (for Stage 2)
DATA_DIR = './data/BraTS2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'
if not os.path.exists(DATA_DIR):
    os.makedirs(os.path.dirname(DATA_DIR), exist_ok=True)
    with zipfile.ZipFile(f'{DRIVE_BASE}/TextBraTS_data.zip', 'r') as zf:
        zf.extractall(os.path.dirname(DATA_DIR))
ET_CACHE = f'{DRIVE_BASE}/et_enriched.zip'
if os.path.exists(ET_CACHE):
    with zipfile.ZipFile(ET_CACHE, 'r') as zf:
        zf.extractall(DATA_DIR)
print(f'BraTS2020 data: {len([d for d in os.listdir(DATA_DIR) if d.startswith("BraTS")])} cases')

def sync_and_tag(tag):
    local_ckpt = os.path.join(REPO_DIR, 'checkpoints')
    if not os.path.exists(local_ckpt):
        return
    for f in glob.glob(os.path.join(local_ckpt, '*.pth')):
        shutil.copy2(f, os.path.join(DRIVE_CKPT, os.path.basename(f)))
    best = os.path.join(local_ckpt, 'best.pth')
    if os.path.exists(best):
        shutil.copy2(best, os.path.join(DRIVE_CKPT, f'best_{tag}.pth'))
        print(f'Tagged: best_{tag}.pth')
    print(f'Synced to {DRIVE_CKPT}')

print('Setup complete')

Mounted at /content/drive
Tue Mar 31 10:56:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             48W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+---------------------

In [ ]:
# Extract BraTS2021 from Drive to local SSD
import os, subprocess
os.chdir(REPO_DIR)

BRATS2021_ZIP = os.path.join(DRIVE_BASE, 'BraTS2021_archive.zip')
BRATS2021_LOCAL = '/content/BraTS2021'

if os.path.exists(os.path.join(BRATS2021_LOCAL, 'train')):
    cases = os.listdir(os.path.join(BRATS2021_LOCAL, 'train'))
    print(f'BraTS2021 already extracted. Train cases: {len(cases)}')
else:
    assert os.path.exists(BRATS2021_ZIP), f'BraTS2021_archive.zip not found on Drive: {BRATS2021_ZIP}'
    print('Extracting BraTS2021 to local SSD...')
    os.makedirs('/content/brats2021_tmp', exist_ok=True)
    !unzip -q {BRATS2021_ZIP} -d /content/brats2021_tmp/
    !tar xf /content/brats2021_tmp/BraTS2021_Training_Data.tar -C /content/brats2021_tmp/
    # Move extracted cases to final location
    os.makedirs(BRATS2021_LOCAL, exist_ok=True)
    !mv /content/brats2021_tmp/BraTS2021_* {BRATS2021_LOCAL}/ 2>/dev/null; true
    print('Preparing train/val split...')
    !python scripts/prepare_brats.py --input {BRATS2021_LOCAL} --output {BRATS2021_LOCAL}
    # Cleanup
    !rm -rf /content/brats2021_tmp
    cases = os.listdir(os.path.join(BRATS2021_LOCAL, 'train'))
    print(f'Done. Train cases: {len(cases)}')

## Stage 1: BraTS2021 Pretrain (No Text)

1251 cases, 200 epochs, --no-text-ratio 1.0

In [ ]:
import os, glob, shutil
os.chdir(REPO_DIR)

for f in glob.glob(os.path.join(REPO_DIR, 'checkpoints', '*.pth')):
    os.remove(f)
print('Cleaned checkpoints')

print('Stage 1: BraTS2021 Pretrain (no text)...')
!python -u train.py \
    --config configs/autoresearch/V8.0_stage1_pretrain.yaml \
    --no-text-ratio 1.0 \
    --grad-accum 1

sync_and_tag('V8.0_stage1')
print('Stage 1 complete!')

## Stage 2: BraTS2020 + TextBraTS Fine-tune

Resume from Stage 1, reset optimizer, add text guidance.

In [ ]:
import os, glob
os.chdir(REPO_DIR)

STAGE1_CKPT = os.path.join(DRIVE_CKPT, 'best_V8.0_stage1.pth')
if not os.path.exists(STAGE1_CKPT):
    STAGE1_CKPT = os.path.join(DRIVE_CKPT, 'best_no_text.pth')
assert os.path.exists(STAGE1_CKPT), f'Stage 1 checkpoint not found: {STAGE1_CKPT}'

for f in glob.glob(os.path.join(REPO_DIR, 'checkpoints', '*.pth')):
    os.remove(f)

print(f'Stage 2: Fine-tune from {STAGE1_CKPT}')
!python -u train.py \
    --config configs/autoresearch/V8.0_stage2_finetune.yaml \
    --resume "{STAGE1_CKPT}" \
    --reset-optimizer \
    --reset-lr \
    --no-text-ratio 0.15 \
    --grad-accum 2

sync_and_tag('V8.0')
print('Stage 2 complete!')

## Evaluation

In [ ]:
import subprocess, re, os
os.chdir(REPO_DIR)

ckpt = os.path.join(DRIVE_CKPT, 'best_V8.0.pth')
if not os.path.exists(ckpt):
    ckpt = os.path.join(REPO_DIR, 'checkpoints', 'best.pth')
if not os.path.exists(ckpt):
    ckpt = os.path.join(DRIVE_CKPT, 'last.pth')
assert os.path.exists(ckpt), f'No checkpoint: {ckpt}'
print(f'Using: {ckpt}')

CONFIG = 'configs/autoresearch/V8.0_stage2_finetune.yaml'
BASELINE = {'ET': 0.7910, 'TC': 0.8560, 'WT': 0.8967, 'Mean': 0.8479}

for name, flags in [('text+TTA', ['--use-text', '--tta']), ('notext+TTA', ['--no-text', '--tta'])]:
    print()
    print('=' * 60)
    print(name)
    print('=' * 60)
    cmd = ['python', '-u', 'evaluate_full.py',
           '--config', CONFIG,
           '--checkpoint', ckpt,
           '--split', 'test', '--overlap', '0.5'] + flags
    ret = subprocess.run(cmd, capture_output=True, text=True)
    for line in ret.stdout.split('\n'):
        if 'dice_' in line or 'hd95_' in line:
            print(f'  {line.strip()}')
    if ret.returncode != 0:
        print(f'ERROR: {ret.stderr[-300:]}')

print()
print(f'Baseline V5.0: ET={BASELINE["ET"]}, TC={BASELINE["TC"]}, WT={BASELINE["WT"]}, Mean={BASELINE["Mean"]}')
print('Target: ET > 0.833 (TextBraTS SOTA)')